# Aurora Forecasts - Technology Processing

## Overview

This notebook processes yearly technology capacity and generation forecasts for all European regions.

### Key Functions:
- Extracts capacity and generation data by technology
- Processes data across all European regions
- Formats for generation tracker
- Enables renewable resource deployment analysis

### Technologies Covered:
Solar, Wind, Nuclear, Gas, Coal, Hydro, Biomass, and others


## 1. Import Dependencies

Import required libraries for technology data processing.

In [1]:
import pandas as pd
from datetime import datetime

In [3]:
# Paths

tracker_path = '../../../data/aurora/processed/generation_tracker.xlsx'
path_technology = '../../../data/aurora/aurora2_technology_scenarios_ES_default_currency_1y.parquet'
sharepoint_folder = "Market Research/Trackers - EUROPE/Generation Mix/Backup/Python"

In [4]:
df_technology2 = pd.read_parquet(path_technology)

In [5]:
df_technology2['type'].unique()

array(['Capacity', 'Generation', 'Marginal technology',
       'New build capacity', 'Uncurtailed capture price',
       'Curtailed capture price (M0)',
       'Capture price curtailed below zero',
       'Curtailment rate below zero',
       'Capture price curtailed - fleet wide',
       'Curtailment rate - fleet wide', 'M0 reference price'],
      dtype=object)

In [6]:
# Get RES capacity forecast for 

from aurora_forecasts.processing.dicts import country_tracker_map, region_tracker_map

df_technology2['country'] = df_technology2['region_code'].map(country_tracker_map, na_action='ignore')
df_technology2['region'] = df_technology2['region_code'].map(region_tracker_map, na_action='ignore')

df_technology2['region'] = df_technology2['region'].replace('GB', 'United Kingdom')

In [7]:
# Change interconnector labels for net exports and assigning negative value to those values

df_technology2['value'] = pd.to_numeric(df_technology2['value'], errors='coerce')

df_technology2.loc[(df_technology2['type'] == 'Generation') & (df_technology2['Subgroup'] == 'Interconnectors'), 'Subgroup'] = 'Net Exports'
df_technology2.loc[df_technology2['Subgroup'] == 'Net Exports', 'value'] = -df_technology2.loc[df_technology2['Subgroup'] == 'Net Exports', 'value']

df_technology2

,year,Group,Subgroup,type,value,unit,name,currency,sensitivity,region_code,country,region
0,2021,Battery storage,Battery storage,Capacity,0.000,GW,Iberia October 2020 (High),eur2019,high,esp,Spain,Spain
1,2021,Battery storage,Battery storage,Generation,0.000,TWh,Iberia October 2020 (High),eur2019,high,esp,Spain,Spain
2,2021,Battery storage,Battery storage,Marginal technology,0.000,% year,Iberia October 2020 (High),eur2019,high,esp,Spain,Spain
3,2021,Battery storage,Battery storage,New build capacity,0.000,GW,Iberia October 2020 (High),eur2019,high,esp,Spain,Spain
4,2021,Coal,Coal,Capacity,2.900,GW,Iberia October 2020 (High),eur2019,high,esp,Spain,Spain
...,...,...,...,...,...,...,...,...,...,...,...,...
1770966,2060,Solar,Tracking solar PV,Curtailment rate - fleet wide,0.003,-,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal,Portugal
1770967,2060,Solar,Tracking solar PV,Curtailment rate below zero,0.001,-,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal,Portugal
1770968,2060,Solar,Tracking solar PV,Generation,22.700,TWh,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal,Portugal
1770969,2060,Solar,Tracking solar PV,New build capacity,0.000,GW,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal,Portugal


In [8]:

df_technology2['value'] = pd.to_numeric(df_technology2['value'], errors='coerce')

# print(df_technology2[df_technology2['Subgroup'].str.lower().str.contains('solar')]['Subgroup'].unique())

# print(df_technology2[(df_technology2['Subgroup'] == 'Solar PV') & (df_technology2['name'].str.lower().str.contains('ibe'))]['name'].unique())

df_technology2_filtered = df_technology2[
    # (df_technology2['country'].isin(country_list)) &
    (df_technology2['type'].isin(['Capacity', 'Generation']))
    # (df_technology2['Group'].isin(['Onshore wind', 'Offshore wind', 'Solar'])) &
    # (df_technology2['sensitivity'] == 'central')
    # (~(df_technology2['Subgroup'].isin(['Solar CSP'])))
    ]


In [10]:
# We change the subgroup 'Interconnector' label to 'Net Exports' for all rows in the 
# DataFrame where the 'Subgroup' column has the value 'Interconnector' for allignment with the generation tracker.

df = df_technology2_filtered.copy()
df['Subgroup'] = df['Subgroup'].replace({'Interconnector': 'Net Exports'})

In [ ]:
# Adapt format to tracker format

from aurora_forecasts.utils.forecast_release import parse_release_info
from aurora_forecasts.processing import convert_units

df_fil = df[
    df['sensitivity'].isin(['central', 'low', 'high'])
].reset_index(drop=True)

# Change units: Capacity from GW → MW; Generation from TWh → GWh
convert_units(df_fil)

df_fil.sort_values(by=['name', 'year', 'Group', 'Subgroup'], inplace=True)

df_fil['Source'] = 'Aurora'
df_fil['Scenario'] = df_fil['sensitivity'].str.capitalize()

# df_fil['Group'] = df_fil['Subgroup']

# Derive structured release info for each row from the release name string
df_fil['release'] = df_fil['name'].apply(parse_release_info)
df_fil['release_quarter'] = df_fil['release'].apply(lambda x: x.quarter)
df_fil['release_month']   = df_fil['release'].apply(lambda x: x.month)
df_fil['string_month']    = df_fil['release'].apply(lambda x: x.month_str)
df_fil['release_year']    = df_fil['release'].apply(lambda x: x.year)
df_fil['release_string']  = df_fil['release_year'].astype(str) + '/' + df_fil['string_month']

# columns for pivoting:
pivot_index = [
    'type', 'unit', 'region', 'Source', 'release_year', 'release_month', 'release_string', 'name',   
    'Scenario', 'Subgroup'
       ]

# All columns except 'value' form the identity key: 'value' is excluded because
# it is the field that differs between duplicate rows and will be aggregated below.
dedup_cols = df_fil.columns.difference(['value']).tolist()

# keep=False marks every copy of a duplicated row (not just the second occurrence),
# so the full duplicate set is captured for aggregation.
df_duplicates = df_fil[df_fil.duplicated(keep=False, subset=dedup_cols)].sort_values(by=dedup_cols)
print(f"There are duplicate entries for forecasts\n {df_duplicates['name'].unique()}\n for the following technologies and years:")
print(df_duplicates)
print("DataFrame columns:", df_fil.columns.tolist())
# Resolve duplicates by summing their values — duplicate rows represent additive split entries.
df_dup_corrected = df_duplicates.groupby(dedup_cols).agg({'value': 'sum'}).reset_index()

# Merge the corrected duplicates back with the non-duplicate rows
df_non_duplicates = df_fil[~df_fil.duplicated(keep=False, subset=dedup_cols)]
df_final = pd.concat([df_non_duplicates, df_dup_corrected], ignore_index=True)

df_final.dropna(axis=0, how='any', inplace=True)

dedup_cols.remove('year')

df_fil_pivotted = df_final.pivot(
    index=dedup_cols,
    columns='year',
    values='value'
).reset_index()

# Pad missing year columns from 2016 onwards to ensure a uniform schema across all scenario vintages.
# 2016 is the earliest forecast year across Aurora releases; older vintages may omit years
# before their publication date, so absent columns are filled with NA.
df_fil_pivotted_columns = df_fil_pivotted.columns.tolist()
for year in range(2016, df_fil['year'].max() + 1):
    if year not in df_fil_pivotted_columns:
        df_fil_pivotted[year] = pd.NA

# Reorder columns: metadata index followed by ascending forecast years
df_fil_pivotted = df_fil_pivotted.reindex(
    columns=pivot_index + list(range(2016, df_fil['year'].max() + 1))
)

df_fil_pivotted.sort_values(
    by=['type', 'region', 'Source', 'release_year', 'release_month', 'Scenario', 'Subgroup'],
    inplace=True
)

df_fil_pivotted.reset_index(drop=True, inplace=True)

df_fil_pivotted

There are duplicate entries for forecasts
 ['Great Britain Oct 23 (Central)' 'Great Britain Oct 23 (High)'
 'Great Britain Oct 23 (Low)']
 for the following technologies and years:
        year      Group            Subgroup        type    value unit  \
265562  2023  Other RES  Biomass/ other RES    Capacity   1700.0   MW   
265563  2023  Other RES  Biomass/ other RES    Capacity   6700.0   MW   
265598  2024  Other RES  Biomass/ other RES    Capacity   1700.0   MW   
265599  2024  Other RES  Biomass/ other RES    Capacity   6800.0   MW   
265634  2025  Other RES  Biomass/ other RES    Capacity   1800.0   MW   
...      ...        ...                 ...         ...      ...  ...   
269561  2058  Other RES  Biomass/ other RES  Generation  24300.0  GWh   
269596  2059  Other RES  Biomass/ other RES  Generation   6200.0  GWh   
269597  2059  Other RES  Biomass/ other RES  Generation  24300.0  GWh   
269632  2060  Other RES  Biomass/ other RES  Generation   6200.0  GWh   
269633  2060  Ot

year,type,unit,region,Source,release_year,release_month,release_string,name,Scenario,Subgroup,...,2051,2052,2053,2054,2055,2056,2057,2058,2059,2060
0,Capacity,MW,France,Aurora,2020,10,2020/Oct,France Oct 2020 (Central),Central,Battery storage,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Capacity,MW,France,Aurora,2020,10,2020/Oct,France Oct 2020 (Central),Central,CCGT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Capacity,MW,France,Aurora,2020,10,2020/Oct,France Oct 2020 (Central),Central,Coal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Capacity,MW,France,Aurora,2020,10,2020/Oct,France Oct 2020 (Central),Central,Hydro,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Capacity,MW,France,Aurora,2020,10,2020/Oct,France Oct 2020 (Central),Central,Interconnectors,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25215,Generation,GWh,United Kingdom,Aurora,2026,4,2026/Apr,Great Britain Q2 26 (Low),Low,Offshore wind,...,187600.0,188700.0,187800.0,184800.0,183600.0,183000.0,184500.0,190000.0,194200.0,201200.0
25216,Generation,GWh,United Kingdom,Aurora,2026,4,2026/Apr,Great Britain Q2 26 (Low),Low,Onshore wind,...,54700.0,55600.0,54900.0,55300.0,54700.0,56100.0,56700.0,57700.0,60100.0,59600.0
25217,Generation,GWh,United Kingdom,Aurora,2026,4,2026/Apr,Great Britain Q2 26 (Low),Low,Other thermal,...,9600.0,9700.0,9600.0,9600.0,9600.0,9600.0,9600.0,9600.0,9600.0,9600.0
25218,Generation,GWh,United Kingdom,Aurora,2026,4,2026/Apr,Great Britain Q2 26 (Low),Low,Pumped storage,...,-1800.0,-1700.0,-1700.0,-1600.0,-1500.0,-1400.0,-1400.0,-1300.0,-1300.0,-1200.0


In [12]:
df_fil_pivotted.to_excel(tracker_path, index=False)

In [ ]:
# Authenticate with Microsoft Graph and upload the tracker Excel to SharePoint.
# Credentials are loaded from YAML config + environment variables — never hardcoded.
from pathlib import Path
import os

sp_base_path = os.getenv("SP_BASE_PATH")  # Example env variable for base path; adjust name as needed

# Extract just the filename from the local tracker path for use as the SharePoint upload target
tracker_path = Path(tracker_path).name

sharepoint_target_path = f"{sp_base_path}/{sharepoint_folder}/{tracker_path}"

df_fil_pivotted.to_excel(sharepoint_target_path, index=False)

RuntimeError: Config file not found at: C:\Users\mpy\Python Unified\packages\common_libs\config\ms_config.yaml
Set MS_CONFIG_PATH to override the location.

: 

: 